# Lab 16: Data Science Agent avec GCP BigQuery

**Navigation** : [Lab 15 <<](../Day6-MLE-Star/Lab15-Kaggle-Challenge.ipynb) | [Index](../../README.md) | [>> Lab 17](Lab17-Final-Project.ipynb)

## Objectifs d'apprentissage

À la fin de ce laboratoire, vous saurez :
1. Comprendre l'architecture de l'agent Data Science Google
2. Explorer NL2SQL et NL2Py pour BigQuery
3. Intégrer BQML pour le machine learning dans le data warehouse
4. Architecturer un agent de production sur le cloud

### Prérequis
- Lab 15 (MLE-STAR) complété
- Compte GCP (optionnel pour ce lab théorique)
- Connaissance de SQL

### Durée estimée : 30-40 minutes

## 1. Configuration

In [1]:
import sys
sys.path.insert(0, '..')

import json
import re
from typing import List, Dict, Optional
from dataclasses import dataclass

from config import get_settings
from utils import LLMClient

print("Imports OK : json, re, dataclasses, config, utils")

Imports OK : json, re, dataclasses, config, utils


Chargement des parametres de configuration.

In [2]:
settings = get_settings()
print(f'Provider: {settings.active_provider}')

Provider: openrouter


## 2. NL2SQL Translator

Le **text-to-SQL** (ou NL2SQL) est la tache de traduire une question en langage naturel vers une
requete SQL executable sur un schema de base de donnees. C'est un probleme classique de semantic
parsing dont le benchmark de reference est **Spider** (Yu et al. 2018) : 10 181 questions et
5 693 requetes SQL complexes sur 200 bases de donnees couvrant 138 domaines, etabli pour mesurer
la generalisation cross-domaine des modeles.

> Reference : Yu, T., Zhang, R., Yang, K., et al. (2018).
> *Spider: A Large-Scale Human-Labeled Dataset for Complex and Cross-Domain Semantic Parsing
> and Text-to-SQL Task*. EMNLP 2018. arXiv:1809.08887. https://arxiv.org/abs/1809.08887

In [1]:
@dataclass
class SQLQuery:
    sql: str
    explanation: str
    tables_used: List[str]

class NL2SQLTranslator:
    def __init__(self, llm: LLMClient):
        self.llm = llm

    def translate(self, question: str, schema: Dict) -> SQLQuery:
        schema_desc = self._format_schema(schema)
        prompt = f"""Convertis cette question en SQL BigQuery.

SCHEMA:
{schema_desc}

QUESTION: {question}

EXPLANATION: [explication]
SQL:
```sql
[requete]
```"""
        response = self.llm.generate(prompt, temperature=0.2)
        explanation = ''
        sql = ''
        if 'EXPLANATION:' in response:
            match = re.search(r'EXPLANATION:\s*(.+?)(?=SQL:|$)', response, re.DOTALL)
            explanation = match.group(1).strip() if match else ''
        sql_match = re.search(r'```sql\s*(.*?)\s*```', response, re.DOTALL)
        sql = sql_match.group(1).strip() if sql_match else ''
        return SQLQuery(sql=sql, explanation=explanation, tables_used=[])

    def _format_schema(self, schema: Dict) -> str:
        return '\n'.join([f'Table {t}: {cols}' for t, cols in schema.items()])

print("Classes definies : SQLQuery (dataclass), NL2SQLTranslator (langage naturel vers SQL BigQuery)")

Classes definies : SQLQuery (dataclass), NL2SQLTranslator (langage naturel vers SQL BigQuery)


## 3. NL2Py Translator

Le **NL2Py** (traduction de langage naturel vers code Python) releve de la generation de code par
LLM. Le benchmark de reference pour evaluer cette capacite est **HumanEval** (Chen et al. 2021),
qui mesure la generation de fonctions Python a partir de descriptions en langage naturel via la
metrique pass@k.

> Reference : Chen, M., Tworek, J., Jun, H., et al. (2021).
> *Evaluating Large Language Models Trained on Code*. arXiv:2107.03374 (OpenAI).
> https://arxiv.org/abs/2107.03374

In [1]:
@dataclass
class PythonCode:
    code: str
    explanation: str

class NL2PyTranslator:
    def __init__(self, llm: LLMClient):
        self.llm = llm

    def translate(self, question: str, data_context: str) -> PythonCode:
        prompt = f"""Genere du code Python pour: {question}

CONTEXTE: {data_context}

EXPLANATION: [explication]
CODE:
```python
[code]
```"""
        response = self.llm.generate(prompt, temperature=0.2)
        explanation = ''
        code = ''
        if 'EXPLANATION:' in response:
            match = re.search(r'EXPLANATION:\s*(.+?)(?=CODE:|$)', response, re.DOTALL)
            explanation = match.group(1).strip() if match else ''
        code_match = re.search(r'```python\s*(.*?)\s*```', response, re.DOTALL)
        code = code_match.group(1).strip() if code_match else ''
        return PythonCode(code=code, explanation=explanation)

print("Classes definies : PythonCode (dataclass), NL2PyTranslator (langage naturel vers code Python)")

Classes definies : PythonCode (dataclass), NL2PyTranslator (langage naturel vers code Python)


## 4. Data Science Agent

In [1]:
class DataScienceAgent:
    def __init__(self):
        self.llm = LLMClient()
        self.nl2sql = NL2SQLTranslator(self.llm)
        self.nl2py = NL2PyTranslator(self.llm)

    def analyze(self, question: str, schema: Dict, mode: str = 'sql') -> Dict:
        print(f'[AGENT] Question: {question}')
        print(f'[AGENT] Mode: {mode}')
        if mode == 'sql':
            result = self.nl2sql.translate(question, schema)
            return {'type': 'sql', 'query': result.sql, 'explanation': result.explanation}
        elif mode == 'python':
            data_context = self.nl2sql._format_schema(schema)
            result = self.nl2py.translate(question, data_context)
            return {'type': 'python', 'code': result.code, 'explanation': result.explanation}
        return {'error': 'Mode non supporte'}

print("Classe DataScienceAgent definie : orchestrateur NL2SQL/NL2Py pour BigQuery")

Classe DataScienceAgent definie : orchestrateur NL2SQL/NL2Py pour BigQuery


## 5. Test avec Schema Simule

In [6]:
schema = {
    'sales': ['date', 'product', 'region', 'quantity', 'revenue'],
    'customers': ['customer_id', 'name', 'segment', 'signup_date'],
    'products': ['product_id', 'name', 'category', 'price']
}

print('Schema BigQuery:')
for table, cols in schema.items():
    print(f'  {table}: {cols}')

Schema BigQuery:
  sales: ['date', 'product', 'region', 'quantity', 'revenue']
  customers: ['customer_id', 'name', 'segment', 'signup_date']
  products: ['product_id', 'name', 'category', 'price']


Test NL2SQL : traduction de requetes naturelles en SQL.

In [7]:
# Test NL2SQL
agent = DataScienceAgent()
question = 'Quel est le revenu total par region?'
result = agent.analyze(question, schema, mode='sql')

print('\\n' + '='*50)
print('RESULTAT NL2SQL:')
print('='*50)
print(f'Explication: {result.get("explanation", "N/A")}')
print(f'SQL: {result.get("query", "N/A")}')

[AGENT] Question: Quel est le revenu total par region?
[AGENT] Mode: sql



\n==================================================
RESULTAT NL2SQL:
Explication: Pour obtenir le revenu total par région, nous devons interroger la table `sales`. Nous sélectionnons la colonne `region` et utilisons la fonction d'agrégation `SUM()` sur la colonne `revenue` pour calculer le revenu total. Enfin, nous utilisons la clause `GROUP BY` pour regrouper les calculs pour chaque région distincte.
SQL: SELECT 
    region, 
    SUM(revenue) AS total_revenue
FROM 
    sales
GROUP BY 
    region;


Provider List: https://docs.litellm.ai/docs/providers



Test NL2Py : generation de code Python a partir de langage naturel.

In [8]:
# Test NL2Py
question2 = 'Calcule la moyenne des revenus par mois'
result2 = agent.analyze(question2, schema, mode='python')

print('\\n' + '='*50)
print('RESULTAT NL2Py:')
print('='*50)
print(f'Explication: {result2.get("explanation", "N/A")}')
print(f'Code: {result2.get("code", "N/A")[:300]}...')

[AGENT] Question: Calcule la moyenne des revenus par mois
[AGENT] Mode: python


\n==================================================
Provider List: https://docs.litellm.ai/docs/providers


RESULTAT NL2Py:
Explication: Pour calculer la moyenne des revenus par mois, nous utilisons la bibliothèque `pandas`. Seule la table `sales` est nécessaire ici, car elle contient à la fois la date et le revenu. La démarche consiste à convertir la colonne `date` en format `datetime`, à extraire la période mensuelle (année-mois) pour regrouper les transactions, puis à appliquer la fonction `mean()` sur la colonne `revenue` pour obtenir la moyenne des transactions pour chaque mois.
Code: import pandas as pd

# Création d'un DataFrame d'exemple basé sur votre contexte (à ignorer si vous avez déjà vos données)
data_sales = {
    'date': ['2023-01-15', '2023-01-20', '2023-02-10', '2023-02-25', '2023-03-05'],
    'product': ['P1', 'P2', 'P1', 'P3', 'P2'],
    'region': ['North', 'South'...


## 6. Resume du Lab
### Architecture Google Data Science Agent
1. NL2SQL: Question naturelle vers BigQuery
2. NL2Py: Question naturelle vers Python
3. BQML: Modeles ML dans BigQuery
### Prochaine etape
Lab 17: Projet Final

## Exercice : Agent Data Science Personnalise

Concevez un agent capable de repondre a des questions sur votre propre schema de donnees.

### Objectifs
1. Definir un schema de donnees realiste
2. Tester NL2SQL et NL2Py avec des questions complexes
3. Analyser les limites de la traduction automatique
4. Proposer des ameliorations

### Instructions



In [9]:
# TODO: Definissez votre schema (ex: e-commerce, sante, finance)
mon_schema = {
    'commandes': ['id', 'client_id', 'date', 'montant', 'statut'],
    'clients': ['id', 'nom', 'email', 'date_inscription', 'segment'],
    'produits': ['id', 'nom', 'categorie', 'prix', 'stock'],
    'lignes_commande': ['id', 'commande_id', 'produit_id', 'quantite', 'prix_unitaire']
}

# TODO: Creez 5 questions de complexite croissante
mes_questions = [
    "Quelle est la commande la plus elevee?",  # Simple
    "...",  # Jointure
    "...",  # Agregation temporelle
    "...",  # Sous-requete
    "..."   # Analyse complexe
]

# TODO: Testez chaque question avec les deux modes
agent = DataScienceAgent()
for q in mes_questions:
    print(f"\\nQUESTION: {q}")
    sql_result = agent.analyze(q, mon_schema, mode='sql')
    py_result = agent.analyze(q, mon_schema, mode='python')
    
    print(f"SQL: {sql_result.get('query', 'N/A')[:100]}...")
    print(f"Python: {py_result.get('code', 'N/A')[:100]}...")

# TODO: Analysez les echecs et proposez des corrections

\nQUESTION: Quelle est la commande la plus elevee?
[AGENT] Question: Quelle est la commande la plus elevee?
[AGENT] Mode: sql


[AGENT] Question: Quelle est la commande la plus elevee?
Provider List: https://docs.litellm.ai/docs/providers


[AGENT] Mode: python


SQL: SELECT *
FROM commandes
ORDER BY montant DESC
LIMIT 1;...
Provider List: https://docs.litellm.ai/docs/providers


Python: import pandas as pd

# Création d'un DataFrame d'exemple basé sur votre contexte 
# (À remplacer par...
\nQUESTION: ...
[AGENT] Question: ...
[AGENT] Mode: sql


[AGENT] Question: ...
Provider List: https://docs.litellm.ai/docs/providers


[AGENT] Mode: python


SQL: SELECT 
    p.categorie,
    SUM(lc.quantite * lc.prix_unitaire) AS chiffre_affaires_total
FROM 
   ...
Provider List: https://docs.litellm.ai/docs/providers


Python: from sqlalchemy import create_engine, Column, Integer, String, Float, Date, ForeignKey
from sqlalche...
\nQUESTION: ...
[AGENT] Question: ...
[AGENT] Mode: sql


...

SQL: SELECT 
    c.segment,
    SUM(cmd.montant) AS chiffre_affaires_total
FROM 
    `votre_projet.votre_...
Provider List: https://docs.litellm.ai/docs/providers


Python: import pandas as pd

def analyser_ventes(df_commandes, df_clients, df_produits, df_lignes_commande):...
\nQUESTION: ...
[AGENT] Question: ...
[AGENT] Mode: sql


[AGENT] Question: ...
Provider List: https://docs.litellm.ai/docs/providers


[AGENT] Mode: python


SQL: SELECT 
    p.categorie,
    SUM(lc.quantite * lc.prix_unitaire) AS chiffre_affaires_total
FROM 
   ...
Provider List: https://docs.litellm.ai/docs/providers


Python: import pandas as pd

# --- 1. Simulation des données (Contexte) ---
# Table clients
clients_df = pd....
\nQUESTION: ...
[AGENT] Question: ...
[AGENT] Mode: sql


[AGENT] Question: ...
Provider List: https://docs.litellm.ai/docs/providers


[AGENT] Mode: python


SQL: SELECT 
    c.segment,
    SUM(cmd.montant) AS chiffre_affaires_total
FROM 
    `ton_projet.ton_data...
Provider List: https://docs.litellm.ai/docs/providers


Python: import pandas as pd

def calculer_ca_par_segment(df_clients, df_commandes):
    """
    Fusionne les...


## Exercice : Comparaison NL2SQL vs NL2Py sur Requetes Complexes

Testez les deux modes de traduction (SQL et Python) sur des requetes de complexite croissante et analysez les forces et faiblesses de chaque approche.

### Objectifs
1. Definir 5 questions de complexite croissante (simple, jointure, agregation, sous-requete, analytique)
2. Traduire chaque question en SQL et en Python
3. Comparer la qualite, la precision et la robustesse des deux traductions

**Indice :**
- SQL est generalement meilleur pour les jointures et agregations
- Python est plus flexible pour les analyses statistiques et visualisations
- Observez a partir de quel niveau de complexite chaque approche commence a echouer

In [10]:
# Exercice : Benchmark NL2SQL vs NL2Py sur requetes de complexite croissante
# Objectif : Identifier les forces et faiblesses de chaque mode de traduction

# Schema de test pour l'exercice
benchmark_schema = {
    'employees': ['emp_id', 'name', 'dept_id', 'salary', 'hire_date'],
    'departments': ['dept_id', 'dept_name', 'location', 'budget'],
    'projects': ['proj_id', 'proj_name', 'dept_id', 'start_date', 'end_date', 'status'],
    'assignments': ['emp_id', 'proj_id', 'hours_worked', 'role']
}

# Questions de complexite croissante
questions_complexite = [
    # Niveau 1: Agregation simple
    "Quel est le salaire moyen par departement?",
    # Niveau 2: Jointure
    "Quels employees travaillent sur des projets en cours dans le departement Engineering?",
    # Niveau 3: Agregation temporelle
    "Combien de projets ont ete crees par trimestre en 2024?",
    # Niveau 4: Sous-requete
    "Quels employees gagnent plus que la moyenne de leur departement?",
    # Niveau 5: Analyse complexe
    "Quel departement a le meilleur ratio budget / heures travaillees sur les projets termines?"
]

agent = DataScienceAgent()

# TODO: Pour chaque question, testez les deux modes et comparez
resultats_comparaison = []
for i, q in enumerate(questions_complexite):
    # TODO etudiant: traduisez en SQL et en Python
    # sql_result = agent.analyze(q, benchmark_schema, mode='sql')
    # py_result = agent.analyze(q, benchmark_schema, mode='python')
    
    resultats_comparaison.append({
        'niveau': i + 1,
        'question': q,
        # 'sql_ok': bool(sql_result.get('query')),
        # 'py_ok': bool(py_result.get('code')),
        # 'sql_query': sql_result.get('query', '')[:80],
        # 'py_code': py_result.get('code', '')[:80]
    })

# TODO: Affichez le tableau comparatif
print("Niveau | Question | SQL OK | Python OK")
print("-------|----------|--------|----------")
for r in resultats_comparaison:
    print(f"  {r['niveau']}    | {r['question'][:40]}... | {'?':>6} | {'?':>8}")

# TODO: Concluez sur les forces de chaque approche
# SQL est meilleur pour: ...
# Python est meilleur pour: ...

print("Exercice a completer : benchmark NL2SQL vs NL2Py")

Exercice a completer


## Exercice : Validation et Correction Automatique du SQL Genere

Les requetes SQL generees par le LLM peuvent contenir des erreurs (noms de colonnes inexacts, jointures manquantes, syntaxe incorrecte). L'objectif est de creer un validateur SQL qui detecte et corrige automatiquement ces problemes.

### Objectifs
1. Implementer une verification des noms de tables et colonnes contre le schema
2. Detecter les jointures manquantes entre tables
3. Proposer des corrections automatiques pour les erreurs detectees

**Indice :**
- Comparez les identifiants dans le SQL avec les noms du schema
- Si une colonne est utilisee sans le prefix de table, suggerez la bonne table
- Les jointures manquantes se detectent quand une clause WHERE ou SELECT reference une colonne d'une table non jointe

In [11]:
# Exercice : Validateur SQL automatique pour les requetes generees par NL2SQL
# Objectif : Detecter et corriger les erreurs dans le SQL genere

class SQLValidator:
    """Validateur de requetes SQL generees par le LLM."""
    
    def __init__(self, schema: dict):
        """
        Args:
            schema: dictionnaire {table: [colonnes]}
        """
        self.schema = schema
        # Construction d'un index colonne -> table(s)
        self.column_index = {}
        for table, cols in schema.items():
            for col in cols:
                if col not in self.column_index:
                    self.column_index[col] = []
                self.column_index[col].append(table)
    
    def validate_tables(self, sql: str) -> list:
        """Verifie que toutes les tables referencees existent dans le schema."""
        # TODO etudiant : extrayez les noms de tables du SQL
        # Indice: cherchez apres FROM, JOIN, UPDATE, INSERT INTO
        errors = []
        # Exemple simplifie:
        for word in sql.split():
            word_clean = word.strip(',;()').lower()
            if word_clean in ['from', 'join']:
                pass  # TODO: verifiez la table suivante
        return errors
    
    def validate_columns(self, sql: str) -> list:
        """Verifie que les colonnes referencees existent dans le schema."""
        # TODO etudiant : extrayez les colonnes et verifiez
        errors = []
        # Indice: cherchez les identifiants apres SELECT, WHERE, GROUP BY, ORDER BY
        return errors
    
    def suggest_fixes(self, sql: str, errors: list) -> str:
        """Propose des corrections pour les erreurs detectees."""
        # TODO etudiant : pour chaque erreur, suggerez une correction
        # Exemple: si 'revenu' n'existe pas, suggerez 'revenue'
        fixed_sql = sql
        return fixed_sql

# TODO: Testez le validateur
# validator = SQLValidator(schema)
# 
# # Requete SQL avec erreurs typiques
# test_sql = """
# SELECT region, SUM(revenu) as total
# FROM sales s
# JOIN customers c ON s.customer_id = c.id
# GROUP BY region
# """
# 
# errors = validator.validate_tables(test_sql)
# errors += validator.validate_columns(test_sql)
# print(f"Erreurs trouvees: {len(errors)}")
# for e in errors:
#     print(f"  - {e}")
# 
# if errors:
#     fixed = validator.suggest_fixes(test_sql, errors)
#     print(f"\nSQL corrige:\n{fixed}")

print("Exercice a completer : validation et correction automatique du SQL genere")

Exercice a completer


### Questions d'analyse
- Quels types de questions echouent le plus souvent ?
- Le mode SQL ou Python est-il plus robuste ?
- Comment ameliorer le contexte fourni au LLM ?


## References

- Yu, T., Zhang, R., Yang, K., et al. (2018). *Spider: A Large-Scale Human-Labeled Dataset for Complex and Cross-Domain Semantic Parsing and Text-to-SQL Task*. EMNLP 2018. arXiv:1809.08887. https://arxiv.org/abs/1809.08887
- Chen, M., Tworek, J., Jun, H., et al. (2021). *Evaluating Large Language Models Trained on Code*. arXiv:2107.03374 (OpenAI). https://arxiv.org/abs/2107.03374
- Xi, Z., et al. (2025). *The Rise and Potential of Large Language Model Based Agents: A Survey*. arXiv:2309.07864.